# Building Your First LLM Helper: From Prompt to Inspectable Python Code

In this lesson, we build a small helper that takes a question about a dataset and returns the result. Here we
Instead of writing analysis code each time, we pass the dataset context and the question to the LLM and let it generate the answer.

## 1 — Setup

We install the necessary libraries, load our API credentials from a `.env` file, and read in the dataset. The completed notebook includes saved outputs so you can review the expected result without my local `.env` file. To rerun the Gemini cells, create your own `.env` file in the project root with `GEMINI_API_KEY=your-gemini-api-key-here`.

In [ ]:
%pip install -qq google-genai pandas scikit-learn matplotlib seaborn python-dotenv


In [1]:
import os
import re
from google import genai
from dotenv import load_dotenv
import pandas as pd

# Load environment variables from a .env file into the environment
load_dotenv()

# Read the Gemini API key from the environment variables
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Create a Gemini client using the API key
# This client will be used to send requests to Gemini models
client = genai.Client(api_key=GEMINI_API_KEY)


In [2]:
df = pd.read_csv("../../data/hr_analytics.csv")
print(df.shape)


(5030, 25)


In [3]:
df.head().T

,0,1,2,3,4
Employee ID,1001,1002,1003,1004,1005
age,27,34,50,31,51
gender,Female,Male,Female,Male,Male
department,Engineering,Engineering,Finance,Marketing,Marketing
department_code,ENG-02,ENG-02,FIN-05,MKT-04,MKT-04
JobTitle,Backend Developer,Data Engineer,Controller,Marketing Analyst,SEO Specialist
job_level,3,2,4,3,4
Education,High School,Master's,Master's,Bachelor's,Master's
MonthlyIncome,8001,8777,14021,6402,9277
monthly_rate,9162,10301,18470,8805,12125


## 2 — A Glimpse of the LLM Helper Function


In [4]:
SYSTEM_PROMPT = (
    "Write pandas code using the existing DataFrame variable `df`. "
    "Use up-to-date pandas 2.x and Python 3.10+ syntax. Avoid deprecated arguments or methods. "
    "Store the final result in `result_df`. "
    "Do NOT create a new DataFrame from scratch. "
    "Do NOT include import statements. "
    "Return ONLY executable Python code, no explanations."
)


def eda_helper(question: str, frame: pd.DataFrame):
    """Ask a plain-English question about df and get back a DataFrame result."""

    # Build the prompt sent to the model.
    # We include the column names so the model knows the schema of the dataset.
    prompt = f"Columns: {list(frame.columns)}\nQuestion: {question}"

    # Send the prompt and system instructions to the Gemini model.
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "temperature": 0.0,  # Makes the output deterministic
            "seed": 42,
            "system_instruction": SYSTEM_PROMPT,  # System level instructions that define rules or behavior for the model
        },
    )
    # Extract executable Python from the model response.
    text = (response.text or "").strip()
    match = re.search(
        r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE
    )
    code = match.group(1).strip() if match else text

    # Define a restricted execution environment.
    # The generated code can only access pandas (pd) and the DataFrame (df).
    env = {"pd": pd, "df": frame.copy()}

    # Execute the generated code in the restricted environment.
    exec(code, env, env)

    # The model is instructed to store the final output in `result_df`.
    result = env.get("result_df")
    if result is None:
        raise RuntimeError("Generated code did not assign `result_df`.")
    return result

## 3 — Let's Try It Out

In [5]:
eda_helper("How many missing values does each column have?", df)

Employee ID                   0
age                           0
gender                        0
department                    0
department_code               0
JobTitle                      0
job_level                     0
Education                     0
MonthlyIncome                 0
monthly_rate                  0
hourly_rate                   0
daily_rate                    0
years_at_company              0
years_in_role                 0
years_since_promotion         0
satisfaction_score            0
environment_satisfaction      0
Attrition                     0
OverTime                      0
distance_from_home           67
training_hours_last_year     64
num_companies_worked          0
manager_rating              460
work_life_balance             0
last_promotion_date          43
dtype: int64

In [6]:
eda_helper("What is the average age by department?", df)

,department,age
0,Engineering,34.057223
1,Finance,34.070492
2,Human Resources,33.419890
3,Marketing,34.148760
4,Operations,34.353749
5,Sales,34.335508


In [7]:
eda_helper("What are the top 10 highest paid employees?", df)

,Employee ID,age,gender,department,department_code,JobTitle,job_level,Education,MonthlyIncome,monthly_rate,...,satisfaction_score,environment_satisfaction,Attrition,OverTime,distance_from_home,training_hours_last_year,num_companies_worked,manager_rating,work_life_balance,last_promotion_date
2306,3307,54,Male,Engineering,ENG-02,ML Engineer,5,Master's,20659,23089,...,3 - High,4,No,No,12.0,20.0,0,3.0,High,2017-07-31
1233,2234,38,Male,Engineering,ENG-02,Staff Engineer,5,PhD,20396,24862,...,2 - Medium,3,No,No,2.0,12.0,1,4.0,Very High,2022-07-02
2461,3462,43,Female,Engineering,ENG-02,ML Engineer,5,Master's,20054,26996,...,3 - High,4,No,No,7.0,42.0,0,4.0,Very High,2007-12-03
4648,5649,24,Female,Engineering,ENG-02,Staff Engineer,5,PhD,19422,22028,...,3 - High,2,No,No,3.0,16.0,3,5.0,Very High,11/28/2024
3856,4857,42,Female,Engineering,ENG-02,Engineering Manager,5,Master's,18350,24309,...,3 - High,4,No,No,5.0,21.0,3,4.0,Very High,08/22/2021
2986,3987,40,Male,Engineering,ENG-02,ML Engineer,5,Bachelor's,18214,24025,...,3 - High,4,No,No,7.0,33.0,0,5.0,Very High,2023-10-12
1089,2090,50,Female,Engineering,ENG-02,DevOps Engineer,5,Bachelor's,17614,23839,...,2 - Medium,3,No,No,3.0,32.0,2,3.0,Medium,2018-03-12
2229,3230,32,Female,Sales,SAL-03,Sales Director,5,PhD,17542,22584,...,4 - Very High,4,No,No,6.0,26.0,0,3.0,High,10/15/2022
3836,4837,46,Female,Sales,SAL-03,Sales Director,5,Bachelor's,17511,21074,...,4 - Very High,3,Yes,Yes,5.0,20.0,2,4.0,Very High,04/05/2021
643,1644,39,Female,Engineering,ENG-02,Staff Engineer,5,Bachelor's,17459,22266,...,2 - Medium,2,Yes,No,19.0,30.0,0,3.0,Very High,2020-04-25


## 4 — Show me the Code

Let's add a `show_code` flag so you can see exactly what the LLM wrote before it runs.

In [8]:
def eda_helper(question, frame: pd.DataFrame, show_code: bool = False):
    """Ask a plain-English question about df; get back a DataFrame."""
    prompt = f"Columns: {list(frame.columns)}\nQuestion: {question}"

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "temperature": 0.0,
            "seed": 42,
            "system_instruction": SYSTEM_PROMPT,
        },
    )

    text = (response.text or "").strip()
    match = re.search(
        r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE
    )
    code = match.group(1).strip() if match else text

    if show_code:
        print("--- Generated Code ---")
        print(code)
        print("---\n")

    env = {"pd": pd, "df": frame.copy()}
    exec(code, env, env)
    result = env.get("result_df")
    if result is None:
        raise RuntimeError("Generated code did not assign `result_df`.")
    return result

In [9]:
eda_helper("How many missing values does each column have?", df, show_code=True)

--- Generated Code ---
result_df = df.isnull().sum()
---



Employee ID                   0
age                           0
gender                        0
department                    0
department_code               0
JobTitle                      0
job_level                     0
Education                     0
MonthlyIncome                 0
monthly_rate                  0
hourly_rate                   0
daily_rate                    0
years_at_company              0
years_in_role                 0
years_since_promotion         0
satisfaction_score            0
environment_satisfaction      0
Attrition                     0
OverTime                      0
distance_from_home           67
training_hours_last_year     64
num_companies_worked          0
manager_rating              460
work_life_balance             0
last_promotion_date          43
dtype: int64

In [10]:
eda_helper("What is the average age by department?", df, show_code=True)

--- Generated Code ---
result_df = df.groupby('department')['age'].mean().reset_index()
---



,department,age
0,Engineering,34.057223
1,Finance,34.070492
2,Human Resources,33.419890
3,Marketing,34.148760
4,Operations,34.353749
5,Sales,34.335508


In [11]:
eda_helper("What are the top 10 highest paid employees?", df, show_code=True)

--- Generated Code ---
result_df = df.nlargest(10, 'MonthlyIncome')
---



,Employee ID,age,gender,department,department_code,JobTitle,job_level,Education,MonthlyIncome,monthly_rate,...,satisfaction_score,environment_satisfaction,Attrition,OverTime,distance_from_home,training_hours_last_year,num_companies_worked,manager_rating,work_life_balance,last_promotion_date
2306,3307,54,Male,Engineering,ENG-02,ML Engineer,5,Master's,20659,23089,...,3 - High,4,No,No,12.0,20.0,0,3.0,High,2017-07-31
1233,2234,38,Male,Engineering,ENG-02,Staff Engineer,5,PhD,20396,24862,...,2 - Medium,3,No,No,2.0,12.0,1,4.0,Very High,2022-07-02
2461,3462,43,Female,Engineering,ENG-02,ML Engineer,5,Master's,20054,26996,...,3 - High,4,No,No,7.0,42.0,0,4.0,Very High,2007-12-03
4648,5649,24,Female,Engineering,ENG-02,Staff Engineer,5,PhD,19422,22028,...,3 - High,2,No,No,3.0,16.0,3,5.0,Very High,11/28/2024
3856,4857,42,Female,Engineering,ENG-02,Engineering Manager,5,Master's,18350,24309,...,3 - High,4,No,No,5.0,21.0,3,4.0,Very High,08/22/2021
2986,3987,40,Male,Engineering,ENG-02,ML Engineer,5,Bachelor's,18214,24025,...,3 - High,4,No,No,7.0,33.0,0,5.0,Very High,2023-10-12
1089,2090,50,Female,Engineering,ENG-02,DevOps Engineer,5,Bachelor's,17614,23839,...,2 - Medium,3,No,No,3.0,32.0,2,3.0,Medium,2018-03-12
2229,3230,32,Female,Sales,SAL-03,Sales Director,5,PhD,17542,22584,...,4 - Very High,4,No,No,6.0,26.0,0,3.0,High,10/15/2022
3836,4837,46,Female,Sales,SAL-03,Sales Director,5,Bachelor's,17511,21074,...,4 - Very High,3,Yes,Yes,5.0,20.0,2,4.0,Very High,04/05/2021
643,1644,39,Female,Engineering,ENG-02,Staff Engineer,5,Bachelor's,17459,22266,...,2 - Medium,2,Yes,No,19.0,30.0,0,3.0,Very High,2020-04-25
